Nuovi dataset
↓
Normalizzazione nomi
↓
Test match rate
↓
Merge
↓
Nuove feature (xG, xA)
↓
V2 dataset

In [37]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [40]:
import src.data_utils

print(src.data_utils.__file__)

dir(src.data_utils)

/Users/Pietromiragoli/github-projects:/champions-league-player-discovery/src/data_utils.py


['__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__']

In [41]:
import importlib
import src.data_utils

importlib.reload(src.data_utils)

dir(src.data_utils)

['__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 'normalize_name',
 're',
 'unicodedata']

In [42]:
from src.data_utils import normalize_name
from src.similarity_engine_v2 import find_similar_non_ucl_players_v3

In [1]:
# ==================================================
# Imports
# ==================================================

import pandas as pd
import numpy as np

In [2]:
# ==================================================
# Load datasets
# ==================================================

fbref_df = pd.read_csv(
    "../data/raw/players_data_light-2025_2026.csv"
)

profiles = pd.read_csv(
    "../data/raw/all_player_profiles.csv"
)

stats = pd.read_csv(
    "../data/raw/all_player_stats.csv"
)

print("FBref:", fbref_df.shape)
print("Profiles:", profiles.shape)
print("Stats:", stats.shape)

FBref: (2779, 53)
Profiles: (4588, 5)
Stats: (3730, 17)


In [3]:
# ==================================================
# Quick inspection
# ==================================================

display(fbref_df.head())

display(profiles.head())

display(stats.head())

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,Save%,W,D,L,CS,CS%,PKatt_stats_keeper,PKA,PKsv,PKm
0,1,Brenden Aaronson,us USA,"MF,FW",Leeds United,eng Premier League,25.0,2000.0,34,27,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Zach Abbott,eng ENG,DF,Nottingham Forest,eng Premier League,19.0,2006.0,3,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Jones El-Abdellaoui,ma MAR,"MF,FW",Celta Vigo,es La Liga,20.0,2006.0,21,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Himad Abdelli,dz ALG,MF,Marseille,fr Ligue 1,26.0,1999.0,8,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Himad Abdelli,dz ALG,MF,Angers,fr Ligue 1,26.0,1999.0,13,11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,player_id,name,league,position,market_value
0,804508,Viktor Gyökeres,Premier League,F,61000000.0
1,934235,Bukayo Saka,Premier League,F,126000000.0
2,794839,Gabriel Jesus,Premier League,F,21000000.0
3,922573,Gabriel Martinelli,Premier League,F,42000000.0
4,836705,Kai Havertz,Premier League,F,48000000.0


,player_id,league,appearances,matches_started,minutes_played,goals,assists,expected_goals,expected_assists,rating,total_shots,shots_on_target,yellow_cards,red_cards,tackles,interceptions,saves
0,804508,Premier League,31,24,2011,12,0,10.3733,1.773005,6.564516,47,18,5,0,6,1,0
1,934235,Premier League,27,22,2001,6,3,6.9941,5.761951,7.214815,63,26,1,0,36,14,0
2,794839,Premier League,12,2,320,2,0,1.8434,0.247789,6.541667,15,10,2,0,4,2,0
3,922573,Premier League,27,10,951,1,3,3.7215,1.012725,6.592593,26,11,2,0,11,1,0
4,836705,Premier League,8,5,408,1,1,2.5193,0.108973,6.637500,12,3,0,0,5,0,0


In [4]:
# ==================================================
# Column inspection
# ==================================================

print("FBref columns:")
print(fbref_df.columns.tolist())

print("\nProfiles columns:")
print(profiles.columns.tolist())

print("\nStats columns:")
print(stats.columns.tolist())

FBref columns:
['Rk', 'Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'Born', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'CrdY', 'CrdR', 'G+A-PK', 'Sh', 'SoT', 'SoT%', 'Sh/90', 'SoT/90', 'G/Sh', 'G/SoT', 'PK_stats_shooting', 'PKatt_stats_shooting', 'Crs', 'TklW', 'Int', 'Fld', 'CrdY_stats_misc', 'CrdR_stats_misc', '2CrdY', 'Fls', 'OG', 'GA', 'GA90', 'SoTA', 'Saves', 'Save%', 'W', 'D', 'L', 'CS', 'CS%', 'PKatt_stats_keeper', 'PKA', 'PKsv', 'PKm']

Profiles columns:
['player_id', 'name', 'league', 'position', 'market_value']

Stats columns:
['player_id', 'league', 'appearances', 'matches_started', 'minutes_played', 'goals', 'assists', 'expected_goals', 'expected_assists', 'rating', 'total_shots', 'shots_on_target', 'yellow_cards', 'red_cards', 'tackles', 'interceptions', 'saves']


In [6]:
import unicodedata
import re

def normalize_name(name):

    name = str(name)

    name = unicodedata.normalize("NFKD", name)
    name = name.encode("ascii", "ignore").decode("utf-8")

    name = name.lower()

    name = re.sub(r"\s+", " ", name)

    return name.strip()

In [7]:
fbref_df["player_clean"] = (
    fbref_df["Player"]
    .apply(normalize_name)
)

profiles["player_clean"] = (
    profiles["name"]
    .apply(normalize_name)
)

In [8]:
fbref_names = set(
    fbref_df["player_clean"]
)

profile_names = set(
    profiles["player_clean"]
)

matches = fbref_names.intersection(profile_names)

print("FBref players:", len(fbref_names))
print("Profiles players:", len(profile_names))
print("Matches:", len(matches))

print(
    "Match rate:",
    round(
        len(matches) / len(fbref_names) * 100,
        2
    ),
    "%"
)

FBref players: 2627
Profiles players: 4566
Matches: 2214
Match rate: 84.28 %


In [9]:
# Merge profiles with advanced stats using player_id and league

player_stats = profiles.merge(
    stats,
    on=["player_id", "league"],
    how="inner"
)

player_stats["player_clean"] = (
    player_stats["name"]
    .apply(normalize_name)
)

player_stats.head()

,player_id,name,league,position,market_value,player_clean,appearances,matches_started,minutes_played,goals,...,expected_goals,expected_assists,rating,total_shots,shots_on_target,yellow_cards,red_cards,tackles,interceptions,saves
0,804508,Viktor Gyökeres,Premier League,F,61000000.0,viktor gyokeres,31,24,2011,12,...,10.3733,1.773005,6.564516,47,18,5,0,6,1,0
1,934235,Bukayo Saka,Premier League,F,126000000.0,bukayo saka,27,22,2001,6,...,6.9941,5.761951,7.214815,63,26,1,0,36,14,0
2,794839,Gabriel Jesus,Premier League,F,21000000.0,gabriel jesus,12,2,320,2,...,1.8434,0.247789,6.541667,15,10,2,0,4,2,0
3,922573,Gabriel Martinelli,Premier League,F,42000000.0,gabriel martinelli,27,10,951,1,...,3.7215,1.012725,6.592593,26,11,2,0,11,1,0
4,836705,Kai Havertz,Premier League,F,48000000.0,kai havertz,8,5,408,1,...,2.5193,0.108973,6.637500,12,3,0,0,5,0,0


In [10]:
# Add expected goals and expected assists to FBref dataset

enhanced_df = fbref_df.merge(
    player_stats[
        [
            "player_clean",
            "expected_goals",
            "expected_assists"
        ]
    ],
    on="player_clean",
    how="left"
)

enhanced_df.head()

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,L,CS,CS%,PKatt_stats_keeper,PKA,PKsv,PKm,player_clean,expected_goals,expected_assists
0,1,Brenden Aaronson,us USA,"MF,FW",Leeds United,eng Premier League,25.0,2000.0,34,27,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,brenden aaronson,4.2381,3.046127
1,2,Zach Abbott,eng ENG,DF,Nottingham Forest,eng Premier League,19.0,2006.0,3,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,zach abbott,NaN,0.009372
2,3,Jones El-Abdellaoui,ma MAR,"MF,FW",Celta Vigo,es La Liga,20.0,2006.0,21,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,jones el-abdellaoui,2.5514,0.616849
3,4,Himad Abdelli,dz ALG,MF,Marseille,fr Ligue 1,26.0,1999.0,8,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,himad abdelli,2.1327,0.782131
4,5,Himad Abdelli,dz ALG,MF,Angers,fr Ligue 1,26.0,1999.0,13,11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,himad abdelli,2.1327,0.782131


In [11]:
enhanced_df[
    ["Player", "Squad", "Comp", "Gls", "Ast", "expected_goals", "expected_assists"]
].head(20)

,Player,Squad,Comp,Gls,Ast,expected_goals,expected_assists
0,Brenden Aaronson,Leeds United,eng Premier League,4,5,4.2381,3.046127
1,Zach Abbott,Nottingham Forest,eng Premier League,0,0,NaN,0.009372
2,Jones El-Abdellaoui,Celta Vigo,es La Liga,2,0,2.5514,0.616849
3,Himad Abdelli,Marseille,fr Ligue 1,0,0,2.1327,0.782131
4,Himad Abdelli,Angers,fr Ligue 1,2,0,2.1327,0.782131
5,Ali Abdi,Nice,fr Ligue 1,3,0,1.6372,0.608200
6,Salis Abdul Samed,Nice,fr Ligue 1,0,0,0.0259,0.133209
7,Saud Abdulhamid,Lens,fr Ligue 1,2,4,0.5177,2.872053
8,Tay Abed,Levante,es La Liga,0,0,NaN,NaN
9,Laurent Abergel,Lorient,fr Ligue 1,1,0,0.7317,0.908153


In [12]:
print(
    "xG coverage:",
    round(enhanced_df["expected_goals"].notna().mean() * 100, 2),
    "%"
)

print(
    "xA coverage:",
    round(enhanced_df["expected_assists"].notna().mean() * 100, 2),
    "%"
)

xG coverage: 73.19 %
xA coverage: 83.73 %


In [13]:
enhanced_df_v2 = enhanced_df.dropna(
    subset=["expected_goals", "expected_assists"]
).copy()

print(enhanced_df_v2.shape)

(2042, 56)


In [14]:
enhanced_df_v2.to_csv(
    "../data/processed/enhanced_player_dataset_v2.csv",
    index=False
)

In [15]:
enhanced_df_v2["xG_per90"] = (
    enhanced_df_v2["expected_goals"]
    / enhanced_df_v2["90s"]
)

enhanced_df_v2["xA_per90"] = (
    enhanced_df_v2["expected_assists"]
    / enhanced_df_v2["90s"]
)

In [16]:
advanced_features = [
    "Gls_per90",
    "Ast_per90",
    "Sh_per90",
    "SoT_per90",
    "G/Sh",
    "xG_per90",
    "xA_per90"
]

In [17]:
# ==================================================
# Advanced features per90
# ==================================================

enhanced_df_v2["xG_per90"] = (
    enhanced_df_v2["expected_goals"]
    / enhanced_df_v2["90s"]
)

enhanced_df_v2["xA_per90"] = (
    enhanced_df_v2["expected_assists"]
    / enhanced_df_v2["90s"]
)

In [18]:
enhanced_df_v2[
    [
        "Player",
        "90s",
        "expected_goals",
        "expected_assists",
        "xG_per90",
        "xA_per90"
    ]
].head()

,Player,90s,expected_goals,expected_assists,xG_per90,xA_per90
0,Brenden Aaronson,24.9,4.2381,3.046127,0.170205,0.122334
2,Jones El-Abdellaoui,7.4,2.5514,0.616849,0.344784,0.083358
3,Himad Abdelli,1.5,2.1327,0.782131,1.421800,0.521421
4,Himad Abdelli,10.5,2.1327,0.782131,0.203114,0.074489
5,Ali Abdi,11.8,1.6372,0.608200,0.138746,0.051542


In [19]:
# ==================================================
# Similarity Engine V2 features
# ==================================================

advanced_features = [
    "Gls_per90",
    "Ast_per90",
    "Sh_per90",
    "SoT_per90",
    "G/Sh",
    "xG_per90",
    "xA_per90"
]

In [21]:
enhanced_df_v2.columns.tolist()

['Rk',
 'Player',
 'Nation',
 'Pos',
 'Squad',
 'Comp',
 'Age',
 'Born',
 'MP',
 'Starts',
 'Min',
 '90s',
 'Gls',
 'Ast',
 'G+A',
 'G-PK',
 'PK',
 'PKatt',
 'CrdY',
 'CrdR',
 'G+A-PK',
 'Sh',
 'SoT',
 'SoT%',
 'Sh/90',
 'SoT/90',
 'G/Sh',
 'G/SoT',
 'PK_stats_shooting',
 'PKatt_stats_shooting',
 'Crs',
 'TklW',
 'Int',
 'Fld',
 'CrdY_stats_misc',
 'CrdR_stats_misc',
 '2CrdY',
 'Fls',
 'OG',
 'GA',
 'GA90',
 'SoTA',
 'Saves',
 'Save%',
 'W',
 'D',
 'L',
 'CS',
 'CS%',
 'PKatt_stats_keeper',
 'PKA',
 'PKsv',
 'PKm',
 'player_clean',
 'expected_goals',
 'expected_assists',
 'xG_per90',
 'xA_per90']

In [22]:
# ==========================================
# Create per90 features
# ==========================================

enhanced_df_v2["Gls_per90"] = (
    enhanced_df_v2["Gls"] /
    enhanced_df_v2["90s"]
)

enhanced_df_v2["Ast_per90"] = (
    enhanced_df_v2["Ast"] /
    enhanced_df_v2["90s"]
)

enhanced_df_v2["Sh_per90"] = (
    enhanced_df_v2["Sh"] /
    enhanced_df_v2["90s"]
)

enhanced_df_v2["SoT_per90"] = (
    enhanced_df_v2["SoT"] /
    enhanced_df_v2["90s"]
)

In [23]:
enhanced_df_v2[
    [
        "Player",
        "Gls_per90",
        "Ast_per90",
        "Sh_per90",
        "SoT_per90",
        "xG_per90",
        "xA_per90"
    ]
].head()

,Player,Gls_per90,Ast_per90,Sh_per90,SoT_per90,xG_per90,xA_per90
0,Brenden Aaronson,0.160643,0.200803,1.726908,0.642570,0.170205,0.122334
2,Jones El-Abdellaoui,0.270270,0.000000,2.297297,0.540541,0.344784,0.083358
3,Himad Abdelli,0.000000,0.000000,2.666667,0.666667,1.421800,0.521421
4,Himad Abdelli,0.190476,0.000000,1.047619,0.476190,0.203114,0.074489
5,Ali Abdi,0.254237,0.000000,0.932203,0.254237,0.138746,0.051542


In [25]:
# ==================================================
# Clean infinite and missing values before scaling
# ==================================================

enhanced_df_v2 = enhanced_df_v2.replace(
    [np.inf, -np.inf],
    np.nan
)

enhanced_df_v2 = enhanced_df_v2.dropna(
    subset=advanced_features
).copy()

print(enhanced_df_v2.shape)

(2016, 62)


In [26]:
X_v2 = enhanced_df_v2[advanced_features]

scaler_v2 = StandardScaler()
X_scaled_v2 = scaler_v2.fit_transform(X_v2)

model_v2 = NearestNeighbors(
    n_neighbors=15,
    metric="euclidean"
)

model_v2.fit(X_scaled_v2)

,n_neighbors,15
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'euclidean'
,p,2
,metric_params,None
,n_jobs,None


In [27]:
julian = enhanced_df_v2[
    enhanced_df_v2["Player"] == "Julián Álvarez"
]

julian[
    [
        "Player",
        "Squad",
        "Gls_per90",
        "Ast_per90",
        "xG_per90",
        "xA_per90"
    ]
]

,Player,Squad,Gls_per90,Ast_per90,xG_per90,xA_per90
92,Julián Álvarez,Atlético Madrid,0.379147,0.189573,0.364716,0.242938


In [28]:
def find_similar_players_v2(
    player_name,
    df,
    model,
    scaler,
    feature_columns,
    n_neighbors=10
):

    player_idx = df[
        df["Player"] == player_name
    ].index[0]

    player_features = df.loc[
        player_idx,
        feature_columns
    ].values.reshape(1, -1)

    player_features_scaled = scaler.transform(
        player_features
    )

    distances, indices = model.kneighbors(
        player_features_scaled,
        n_neighbors=n_neighbors + 1
    )

    similar_players = df.iloc[
        indices[0][1:]
    ].copy()

    similar_players["distance"] = distances[0][1:]

    return similar_players

In [29]:
results_v2 = find_similar_players_v2(
    "Julián Álvarez",
    enhanced_df_v2,
    model_v2,
    scaler_v2,
    advanced_features,
    n_neighbors=10
)

results_v2[
    [
        "Player",
        "Squad",
        "Age",
        "distance",
        "Gls_per90",
        "Ast_per90",
        "xG_per90",
        "xA_per90"
    ]
]

/Users/Pietromiragoli/Library/Python/3.10/lib/python/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


,Player,Squad,Age,distance,Gls_per90,Ast_per90,xG_per90,xA_per90
2271,Bukayo Saka,Arsenal,24.0,0.508145,0.307018,0.175439,0.306759,0.252717
2530,Florian Thauvin,Lens,33.0,0.541849,0.383142,0.191571,0.401429,0.327598
2742,Kenan Yıldız,Juventus,21.0,0.570819,0.338983,0.203390,0.265278,0.307327
2224,Rômulo,RB Leipzig,24.0,0.629091,0.389610,0.173160,0.537104,0.121190
19,Ragnar Ache,Köln,27.0,0.683307,0.366492,0.209424,0.531298,0.054516
1437,Rafael Leão,Milan,26.0,0.723165,0.456853,0.152284,0.489883,0.143236
21,Akor Adams,Sevilla,26.0,0.725263,0.390244,0.146341,0.469239,0.047920
2045,Nicolas Pépé,Villarreal,30.0,0.739321,0.325203,0.243902,0.267427,0.211662
2712,Nico Williams,Athletic Club,23.0,0.773415,0.329670,0.164835,0.259225,0.258248
1070,Gorka Guruzeta,Athletic Club,29.0,0.780446,0.387931,0.129310,0.374591,0.071374


In [31]:
ucl_teams = [
    "Arsenal",
    "Atlético Madrid",
    "Barcelona",
    "Bayern Munich",
    "Benfica",
    "Borussia Dortmund",
    "Club Brugge",
    "Inter",
    "Juventus",
    "Leverkusen",
    "Liverpool",
    "Lille",
    "Manchester City",
    "Milan",
    "Monaco",
    "Napoli",
    "Newcastle Utd",
    "Paris S-G",
    "PSV Eindhoven",
    "Real Madrid",
    "Sporting CP",
    "Tottenham",
    "RB Salzburg"
]

In [32]:
[c for c in enhanced_df_v2.columns if "ucl" in c.lower()]

[]

In [33]:
results_v2["is_ucl_team"] = (
    results_v2["Squad"].isin(ucl_teams)
)

non_ucl_results_v2 = results_v2[
    results_v2["is_ucl_team"] == False
].copy()

non_ucl_results_v2[
    [
        "Player",
        "Squad",
        "Age",
        "distance",
        "Gls_per90",
        "Ast_per90",
        "xG_per90",
        "xA_per90"
    ]
]

,Player,Squad,Age,distance,Gls_per90,Ast_per90,xG_per90,xA_per90
2530,Florian Thauvin,Lens,33.0,0.541849,0.383142,0.191571,0.401429,0.327598
2224,Rômulo,RB Leipzig,24.0,0.629091,0.389610,0.173160,0.537104,0.121190
19,Ragnar Ache,Köln,27.0,0.683307,0.366492,0.209424,0.531298,0.054516
21,Akor Adams,Sevilla,26.0,0.725263,0.390244,0.146341,0.469239,0.047920
2045,Nicolas Pépé,Villarreal,30.0,0.739321,0.325203,0.243902,0.267427,0.211662
2712,Nico Williams,Athletic Club,23.0,0.773415,0.329670,0.164835,0.259225,0.258248
1070,Gorka Guruzeta,Athletic Club,29.0,0.780446,0.387931,0.129310,0.374591,0.071374


In [34]:
def find_similar_non_ucl_players_v3(
    player_name,
    df,
    model,
    scaler,
    feature_columns,
    ucl_teams,
    n_neighbors=30,
    age_range=3
):
    target_player = df[df["Player"] == player_name]

    if target_player.empty:
        raise ValueError(f"Player '{player_name}' not found.")

    target_index = target_player.index[0]
    target_age = target_player["Age"].iloc[0]

    player_features = df.loc[
        target_index,
        feature_columns
    ].values.reshape(1, -1)

    player_features_scaled = scaler.transform(player_features)

    distances, indices = model.kneighbors(
        player_features_scaled,
        n_neighbors=n_neighbors + 1
    )

    similar_players = df.iloc[indices[0][1:]].copy()
    similar_players["distance"] = distances[0][1:]

    similar_players["similarity_score"] = (
        100 / (1 + similar_players["distance"])
    )

    similar_players["is_ucl_team"] = (
        similar_players["Squad"].isin(ucl_teams)
    )

    similar_players = similar_players[
        similar_players["is_ucl_team"] == False
    ].copy()

    similar_players = similar_players[
        similar_players["Age"].between(
            target_age - age_range,
            target_age + age_range
        )
    ].copy()

    return similar_players.sort_values(
        "similarity_score",
        ascending=False
    )

In [35]:
results_v3 = find_similar_non_ucl_players_v3(
    "Julián Álvarez",
    enhanced_df_v2,
    model_v2,
    scaler_v2,
    advanced_features,
    ucl_teams,
    n_neighbors=30,
    age_range=3
)

results_v3[
    [
        "Player",
        "Squad",
        "Age",
        "distance",
        "similarity_score",
        "Gls_per90",
        "Ast_per90",
        "xG_per90",
        "xA_per90"
    ]
]

/Users/Pietromiragoli/Library/Python/3.10/lib/python/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


,Player,Squad,Age,distance,similarity_score,Gls_per90,Ast_per90,xG_per90,xA_per90
2224,Rômulo,RB Leipzig,24.0,0.629091,61.383918,0.389610,0.173160,0.537104,0.121190
19,Ragnar Ache,Köln,27.0,0.683307,59.406882,0.366492,0.209424,0.531298,0.054516
21,Akor Adams,Sevilla,26.0,0.725263,57.962162,0.390244,0.146341,0.469239,0.047920
2712,Nico Williams,Athletic Club,23.0,0.773415,56.388394,0.329670,0.164835,0.259225,0.258248
1070,Gorka Guruzeta,Athletic Club,29.0,0.780446,56.165701,0.387931,0.129310,0.374591,0.071374
1671,Bryan Mbeumo,Manchester Utd,26.0,0.799472,55.571861,0.332103,0.110701,0.304804,0.167405
2440,Sambou Soumano,Lorient,25.0,0.812136,55.183481,0.388350,0.194175,0.521068,0.076225
1997,Igor Paixão,Marseille,25.0,0.842151,54.284367,0.297030,0.247525,0.290045,0.120796
2402,Lassine Sinayoko,Auxerre,26.0,0.844989,54.200865,0.312500,0.138889,0.368066,0.095611
107,Mohamed Amoura,Wolfsburg,26.0,0.879031,53.218917,0.375587,0.140845,0.396488,0.116811


In [36]:
results_v3.to_csv(
    "../data/processed/julian_alvarez_v3_replacements.csv",
    index=False
)

results_v3.head()

,Rk,Player,Nation,Pos,Squad,Comp,Age,Born,MP,Starts,...,expected_assists,xG_per90,xA_per90,Gls_per90,Ast_per90,Sh_per90,SoT_per90,distance,similarity_score,is_ucl_team
2224,2219,Rômulo,br BRA,FW,RB Leipzig,de Bundesliga,24.0,2002.0,28,26,...,2.799485,0.537104,0.121190,0.389610,0.173160,2.640693,1.168831,0.629091,61.383918,False
19,20,Ragnar Ache,de GER,FW,Köln,de Bundesliga,27.0,1998.0,29,18,...,1.041249,0.531298,0.054516,0.366492,0.209424,2.722513,1.151832,0.683307,59.406882,False
21,22,Akor Adams,ng NGA,FW,Sevilla,es La Liga,26.0,2000.0,28,19,...,0.982352,0.469239,0.047920,0.390244,0.146341,2.682927,1.317073,0.725263,57.962162,False
2712,2702,Nico Williams,es ESP,MF,Athletic Club,es La Liga,23.0,2002.0,24,19,...,4.700121,0.259225,0.258248,0.329670,0.164835,2.472527,0.879121,0.773415,56.388394,False
1070,1069,Gorka Guruzeta,es ESP,FW,Athletic Club,es La Liga,29.0,1996.0,31,25,...,1.655867,0.374591,0.071374,0.387931,0.129310,3.017241,1.206897,0.780446,56.165701,False


In [43]:
from src.data_utils import normalize_name

print(normalize_name("Julián Álvarez"))

julian alvarez


In [44]:
from src.data_utils import normalize_name
from src.scouting_score import add_scouting_score
from src.similarity_engine_v2 import find_similar_non_ucl_players_v3

print("All imports successful")

All imports successful
